# University Data Cleaning (Merged)

This notebook merges the two university-cleaning notebooks into one rerunnable flow.

It keeps:
- the stronger NUS / NTU / SUTD cleaning logic from the existing notebooks
- the course-skill columns required by downstream analysis
- a cleaner validation and save flow

It also avoids the original rerun issue where skill extraction depended on an online
model download.


## Setup

The merged notebook uses `SKILL_EXTRACTION_MODE = "auto"`:

- reuse existing saved course skills from `data/cleaned_data/` first, then fall back to the older root-level file if needed
- otherwise recompute skills with the cached local MiniLM model
- if the local model is unavailable, fall back to deterministic keyword matching


In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer, util


DATA_ROOT = Path("../../data")
CLEANED_OUTPUT_DIR = DATA_ROOT / "cleaned_data"
OUTPUT_PATH = CLEANED_OUTPUT_DIR / "combined_courses_cleaned.pkl"
EXISTING_SKILL_SOURCE_PATHS = [
    OUTPUT_PATH,
    DATA_ROOT / "combined_courses_cleaned.pkl",
]

SKILL_EXTRACTION_MODE = "auto"
SKILL_TOP_K = 5
SKILL_MIN_SCORE = 0.30

COURSE_KEY_COLUMNS = ["code", "title", "department", "description", "university"]
POSTGRAD_TITLE_PATTERN = re.compile(
    r"\b(?:postgraduate|post-graduate|doctoral|doctorate|ph\.?d|graduate diploma|graduate certificate|master(?:'|’)s project|master(?:'|’)s thesis|master of computing project)\b",
    re.IGNORECASE,
)
POSTGRAD_DESCRIPTION_PATTERN = re.compile(
    r"(?:cross[- ]listed as a\s+post(?:graduate|grad) course|as a\s+graduate level course|open to\s+ma,\s*ph\.?d)",
    re.IGNORECASE,
)

SPELLING_REPLACEMENTS = {
    "optimisation": "optimization",
    "organisation": "organization",
    "behaviour": "behavior",
    "labour": "labor",
    "analyse": "analyze",
    "analysing": "analyzing",
    "modelling": "modeling",
    "colour": "color",
    "favour": "favor",
    "centre": "center",
    "theatre": "theater",
    "programme": "program",
}


def normalize_spelling(text):
    if not isinstance(text, str):
        return text

    for old, new in SPELLING_REPLACEMENTS.items():
        text = text.replace(old, new)

    return text


def clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()
    text = normalize_spelling(text)
    text = BeautifulSoup(text, "html.parser").get_text(separator=" ")
    text = re.sub(r"[\x00-\x08\x0B-\x0C\x0E-\x1F]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def split_into_chunks(text):
    if not isinstance(text, str) or not text.strip():
        return []

    chunks = re.split(r"[.;:]\s+|\.\s+|\n+", text)
    return [chunk.strip() for chunk in chunks if chunk.strip()]


def safe_word_count(text):
    if not isinstance(text, str):
        return 0
    return len(text.split())


def keep_undergraduate_courses(df):
    title_mask = df["title"].astype(str).str.contains(
        POSTGRAD_TITLE_PATTERN,
        na=False,
    )
    description_mask = df["description"].astype(str).str.contains(
        POSTGRAD_DESCRIPTION_PATTERN,
        na=False,
    )
    course_type_mask = pd.Series(False, index=df.index)
    if "course_type" in df.columns:
        course_type_mask = df["course_type"].astype(str).str.contains(
            POSTGRAD_TITLE_PATTERN,
            na=False,
        )

    return df[~(title_mask | description_mask | course_type_mask)].copy()


def sanitize_string_columns(df):
    illegal_chars_re = re.compile(r"[\x00-\x08\x0B-\x0C\x0E-\x1F]")
    sanitized_df = df.copy()
    for column in sanitized_df.columns:
        sanitized_df[column] = sanitized_df[column].apply(
            lambda value: illegal_chars_re.sub(" ", value)
            if isinstance(value, str)
            else value
        )
    return sanitized_df


def normalize_skill_list(skills):
    return [normalize_spelling(skill.lower()) for skill in skills]


## Load And Clean NUS / NTU / SUTD Data


In [ ]:
def load_nus_courses(data_root=DATA_ROOT):
    with open(data_root / "NUSModsInfo.json", "r", encoding="utf-8") as handle:
        nus_data = json.load(handle)

    nus_fields = ["department", "description", "faculty", "moduleCode", "title"]
    nus_df = pd.DataFrame(
        [{key: item.get(key) for key in nus_fields} for item in nus_data]
    )

    nus_df = nus_df[
        nus_df["moduleCode"].astype(str).str.match(r"^[A-Z]+[0-4]")
    ].copy()

    excluded_faculties = [
        "Temasek Defence Sys. Institute",
        "Cont and Lifelong Education",
        "Duke-NUS Medical School",
    ]
    nus_df = nus_df[~nus_df["faculty"].isin(excluded_faculties)].copy()
    nus_df = nus_df[nus_df["description"].apply(safe_word_count) >= 10].copy()
    nus_df = keep_undergraduate_courses(nus_df)

    nus_df["code"] = nus_df["moduleCode"]
    nus_df["university"] = "NUS"

    return nus_df[["code", "title", "department", "description", "university"]]


def load_ntu_courses(data_root=DATA_ROOT):
    with open(data_root / "ntuCourseInfo.json", "r", encoding="utf-8") as handle:
        ntu_data = json.load(handle)

    ntu_fields = ["code", "title", "description", "departmentMaintaining"]
    ntu_df = pd.DataFrame(
        [{key: item.get(key) for key in ntu_fields} for item in ntu_data]
    )

    ntu_df = ntu_df[
        ntu_df["code"].astype(str).str.match(r"^[A-Z]+[0-4]")
    ].copy()

    dept_mapping_df = pd.read_excel(data_root / "ntu_dept_mapping.xlsx")
    ntu_df = ntu_df.merge(
        dept_mapping_df[["short", "department"]],
        left_on="departmentMaintaining",
        right_on="short",
        how="left",
    ).drop(columns=["short"])

    ntu_df = ntu_df.rename(columns={"departmentMaintaining": "dept_code"})
    ntu_df = ntu_df[ntu_df["description"].apply(safe_word_count) >= 10].copy()
    ntu_df = keep_undergraduate_courses(ntu_df)
    ntu_df["university"] = "NTU"

    return ntu_df[["code", "title", "department", "description", "university"]]


def load_sutd_courses(data_root=DATA_ROOT):
    with open(
        data_root / "sutdCourseDescriptions.json",
        "r",
        encoding="utf-8",
    ) as handle:
        sutd_data = json.load(handle)

    sutd_fields = ["code", "title", "course_type", "description", "affiliations"]
    sutd_df = pd.DataFrame(
        [{key: item.get(key) for key in sutd_fields} for item in sutd_data]
    )

    affiliation_names = {
        "ASD": "Architecture and Sustainable Design",
        "DAI": "Design and Artificial Intelligence",
        "EPD": "Engineering Product Development",
        "ESD": "Engineering Systems and Design",
        "HASS": "Humanities, Arts and Social Sciences",
        "ISTD": "Information Systems Technology and Design",
        "SMT": "Science, Mathematics and Technology",
    }

    sutd_df["department"] = sutd_df["affiliations"].apply(
        lambda codes: [
            affiliation_names[code]
            for code in (codes or [])
            if code in affiliation_names
        ]
    )
    sutd_df["department"] = sutd_df["department"].apply(
        lambda value: value[0] if isinstance(value, list) and value else None
    )
    sutd_df = sutd_df[sutd_df["description"].apply(safe_word_count) >= 10].copy()
    sutd_df = keep_undergraduate_courses(sutd_df)
    sutd_df["university"] = "SUTD"

    return sutd_df[["code", "title", "department", "description", "university"]]


nus_courses_df = load_nus_courses()
ntu_courses_df = load_ntu_courses()
sutd_courses_df = load_sutd_courses()

print("University counts after cleaning:")
print(
    {
        "NUS": len(nus_courses_df),
        "NTU": len(ntu_courses_df),
        "SUTD": len(sutd_courses_df),
    }
)


## Combine And Validate Course Rows

This stage keeps the downstream-friendly course schema used by the analysis notebook,
then adds the cleaner validation summary from the more recent notebook.


In [ ]:
combined_df = pd.concat(
    [nus_courses_df, ntu_courses_df, sutd_courses_df],
    ignore_index=True,
)
combined_df = sanitize_string_columns(combined_df)
combined_df["department"] = combined_df["department"].astype(str).str.strip()
combined_df.insert(0, "s/n", range(1, len(combined_df) + 1))

print("\n=== Courses Validation ===")
print(f"Shape: {combined_df.shape}")
print(f"Null counts:\n{combined_df.isnull().sum()}")
print(f"Universities: {combined_df['university'].value_counts().to_dict()}")

courses_without_description = (
    combined_df["description"].isna()
    | (combined_df["description"].astype(str).str.strip() == "")
).sum()
print(f"Courses without description: {courses_without_description}")

combined_df.head(2)


## Text Preparation


In [ ]:
combined_df["description_clean"] = combined_df["description"].apply(clean_text)
combined_df["description_chunks"] = combined_df["description_clean"].apply(
    split_into_chunks
)

print(
    "Average chunks per course:",
    round(combined_df["description_chunks"].apply(len).mean(), 2),
)


## Skill Vocabulary

The skill vocabulary comes from `data_cleaning_uni.ipynb`, while the merged notebook
adds a more robust extraction flow that can reuse existing saved skills or recompute
them offline.


In [ ]:
hard_skills = normalize_skill_list(
    [
        "python",
        "sql",
        "java",
        "c",
        "c++",
        "programming",
        "machine learning",
        "deep learning",
        "natural language processing",
        "data analysis",
        "data visualization",
        "statistics",
        "probability",
        "linear algebra",
        "optimization",
        "regression analysis",
        "database systems",
        "computer vision",
        "cloud computing",
        "cybersecurity",
        "game theory",
        "simulation modeling",
        "modeling",
        "accounting",
        "managerial accounting",
        "financial accounting",
        "auditing",
        "audit planning",
        "risk assessment",
        "internal controls",
        "financial reporting",
        "financial statement analysis",
        "corporate governance",
        "strategy formulation",
        "strategy implementation",
        "industry analysis",
        "portfolio management",
        "investment banking",
        "financial planning",
        "marketing planning",
        "marketing strategy",
        "project management",
        "budgeting",
        "capital structure",
        "financial markets",
        "market efficiency",
        "credit management",
        "visual analysis",
        "comparative analysis",
        "close reading",
        "literary analysis",
        "textual analysis",
        "historical analysis",
        "critical analysis",
        "archival research",
        "curatorial practice",
        "curatorial methodologies",
        "exhibition studies",
        "art historical methods",
        "ethnography",
        "field ethnography",
        "fieldwork",
        "cultural analysis",
        "social analysis",
        "policy analysis",
        "translation",
        "translation practice",
        "rhetorical analysis",
        "discourse analysis",
        "phonological analysis",
        "textual criticism",
        "comparative study",
        "source analysis",
        "interdisciplinary analysis",
        "historical interpretation",
        "architectural design",
        "landscape design",
        "urban design",
        "architectural history",
        "architectural theory",
        "urban planning",
        "heritage conservation",
        "heritage management",
        "site analysis",
        "mapping",
        "architectural lighting",
        "architectural acoustics",
        "passive design",
        "environmentally responsive design",
        "climate responsive design",
        "ecological design",
        "structural mechanics",
        "structural analysis",
        "building systems",
        "building services systems",
        "energy efficiency",
        "material analysis",
        "signal processing",
        "medical instrumentation",
        "thermodynamic analysis",
        "kinetic analysis",
        "biomechanics",
        "tissue engineering",
        "drug delivery",
        "microfabrication",
        "biosensors",
        "genetic engineering",
        "computational modeling",
        "control theory",
        "robot kinematics",
        "motion tracking",
        "finite element analysis",
        "wastewater treatment",
        "transport infrastructure design",
        "stability assessment",
        "groundwater seepage analysis",
        "structural stability",
        "performance skills",
        "dance performance",
        "music performance",
        "choreography",
        "news writing",
        "media production",
    ]
)

soft_skills = normalize_skill_list(
    [
        "communication",
        "leadership",
        "teamwork",
        "collaboration",
        "critical thinking",
        "problem solving",
        "decision-making",
        "strategic thinking",
        "professional conduct",
        "ethics",
        "ethical responsibility",
        "creativity",
        "adaptability",
        "analytical skills",
        "presentation skills",
        "research skills",
        "writing skills",
        "close reading",
        "interpretation",
        "fieldwork skills",
        "curatorial practice",
        "stylistic sensitivity",
    ]
)

all_skills = hard_skills + soft_skills
hard_skill_set = set(hard_skills)
soft_skill_set = set(soft_skills)


In [ ]:
def attach_existing_skill_matches(df, source_paths=EXISTING_SKILL_SOURCE_PATHS):
    required_columns = COURSE_KEY_COLUMNS + ["skills_embedding"]

    for source_path in source_paths:
        if not source_path.exists():
            continue

        existing_df = pd.read_pickle(source_path)
        if not set(required_columns).issubset(existing_df.columns):
            continue

        existing_skill_df = existing_df[required_columns].drop_duplicates(
            subset=COURSE_KEY_COLUMNS
        )
        hydrated_df = df.merge(
            existing_skill_df,
            on=COURSE_KEY_COLUMNS,
            how="left",
        )

        matched_rows = hydrated_df["skills_embedding"].apply(
            lambda value: isinstance(value, list)
        ).sum()

        if matched_rows:
            print(f"Reused saved skills from: {source_path.resolve()}")
            return hydrated_df, matched_rows

    df["skills_embedding"] = pd.Series([None] * len(df), dtype="object")
    return df, 0


def build_chunk_score_lookup(skill_list, chunk_lists):
    unique_chunks = sorted(
        {
            chunk
            for chunks in chunk_lists
            for chunk in chunks
            if isinstance(chunk, str) and chunk.strip()
        }
    )

    if not unique_chunks:
        return {}, None

    model = SentenceTransformer("all-MiniLM-L6-v2", local_files_only=True)
    skill_embeddings = model.encode(
        skill_list,
        convert_to_tensor=True,
        show_progress_bar=False,
    )

    lookup = {}
    for start in range(0, len(unique_chunks), 256):
        batch = unique_chunks[start : start + 256]
        batch_embeddings = model.encode(
            batch,
            convert_to_tensor=True,
            batch_size=64,
            show_progress_bar=False,
        )
        similarity_matrix = (
            util.cos_sim(batch_embeddings, skill_embeddings)
            .cpu()
            .numpy()
            .astype("float32")
        )

        for chunk, scores in zip(batch, similarity_matrix):
            lookup[chunk] = scores

    return lookup, model


def extract_skills_with_local_embeddings(chunk_lists, skill_list):
    chunk_lookup, _ = build_chunk_score_lookup(skill_list, chunk_lists)
    extracted_skills = []

    for chunks in chunk_lists:
        if not chunks:
            extracted_skills.append([])
            continue

        score_rows = [chunk_lookup[chunk] for chunk in chunks if chunk in chunk_lookup]
        if not score_rows:
            extracted_skills.append([])
            continue

        max_scores = np.vstack(score_rows).max(axis=0)
        scored_skills = [
            (skill_list[i], float(max_scores[i]))
            for i in range(len(skill_list))
            if float(max_scores[i]) >= SKILL_MIN_SCORE
        ]
        scored_skills.sort(key=lambda item: item[1], reverse=True)
        extracted_skills.append(
            [skill for skill, _score in scored_skills[:SKILL_TOP_K]]
        )

    return extracted_skills


def build_keyword_skill_patterns(skill_list):
    unique_skills = sorted(set(skill_list), key=lambda skill: (-len(skill.split()), -len(skill), skill))
    return [
        (
            skill,
            re.compile(rf"(?<!\w){re.escape(skill)}(?!\w)")
        )
        for skill in unique_skills
    ]


def extract_skills_with_keyword_matching(chunk_lists, skill_patterns):
    extracted_skills = []

    for chunks in chunk_lists:
        scores = {}

        for skill, pattern in skill_patterns:
            match_count = sum(1 for chunk in chunks if pattern.search(chunk))
            if match_count:
                scores[skill] = (
                    match_count,
                    len(skill.split()),
                    len(skill),
                )

        ranked_skills = [
            skill
            for skill, _score in sorted(
                scores.items(),
                key=lambda item: item[1],
                reverse=True,
            )
        ]
        extracted_skills.append(ranked_skills[:SKILL_TOP_K])

    return extracted_skills


combined_df, reused_rows = attach_existing_skill_matches(combined_df)

missing_skill_mask = ~combined_df["skills_embedding"].apply(
    lambda value: isinstance(value, list)
)
missing_rows = int(missing_skill_mask.sum())

local_embedding_rows = 0
keyword_fallback_rows = 0

if missing_rows and SKILL_EXTRACTION_MODE in {"auto", "local_embeddings"}:
    try:
        local_skills = extract_skills_with_local_embeddings(
            combined_df.loc[missing_skill_mask, "description_chunks"].tolist(),
            all_skills,
        )
        combined_df.loc[missing_skill_mask, "skills_embedding"] = local_skills
        local_embedding_rows = len(local_skills)
        missing_skill_mask = ~combined_df["skills_embedding"].apply(
            lambda value: isinstance(value, list)
        )
    except Exception as error:
        print(f"Local embedding extraction unavailable: {error}")

if missing_skill_mask.any():
    keyword_patterns = build_keyword_skill_patterns(all_skills)
    fallback_skills = extract_skills_with_keyword_matching(
        combined_df.loc[missing_skill_mask, "description_chunks"].tolist(),
        keyword_patterns,
    )
    combined_df.loc[missing_skill_mask, "skills_embedding"] = fallback_skills
    keyword_fallback_rows = len(fallback_skills)

combined_df["skills_embedding"] = combined_df["skills_embedding"].apply(
    lambda value: value if isinstance(value, list) else []
)
combined_df["hard_skills"] = combined_df["skills_embedding"].apply(
    lambda skills: [skill for skill in skills if skill in hard_skill_set]
)
combined_df["soft_skills"] = combined_df["skills_embedding"].apply(
    lambda skills: [skill for skill in skills if skill in soft_skill_set]
)
combined_df["num_skills"] = combined_df["skills_embedding"].apply(len)
combined_df["num_hard_skills"] = combined_df["hard_skills"].apply(len)
combined_df["num_soft_skills"] = combined_df["soft_skills"].apply(len)

print("Skill extraction summary:")
print(f"Reused existing saved skills: {reused_rows:,}")
print(f"Computed with local embeddings: {local_embedding_rows:,}")
print(f"Computed with keyword fallback: {keyword_fallback_rows:,}")
print(f"Final courses with skill lists: {len(combined_df):,}")


## Save Output

The merged notebook saves the cleaned course dataset to
`../../data/cleaned_data/combined_courses_cleaned.pkl`.


In [ ]:
CLEANED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

combined_df.to_pickle(OUTPUT_PATH)

print(f"Saved cleaned output to: {OUTPUT_PATH.resolve()}")
print(f"Final shape: {combined_df.shape}")
print(f"University counts: {combined_df['university'].value_counts().to_dict()}")

combined_df.head(2)
